# Manual bakery review and finalising classification

In [3]:
from pathlib import Path
from shutil import copy2

import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)


DATA_FOLDER = Path("../data/business/interim")
VERIFICATION_FOLDER = DATA_FOLDER / "ai_verification"
MANUAL_REVIEW_FOLDER = VERIFICATION_FOLDER / "manual_review"
MANUAL_REVIEW_FOLDER.mkdir(parents=True, exist_ok=True)


AI_RESULTS_PATH = VERIFICATION_FOLDER / "bakery_ai_verification_results_v5_final.csv"
REVIEW_PATH = VERIFICATION_FOLDER / "bakery_list_review.csv"

FINAL_RESULTS_PATH = VERIFICATION_FOLDER / "bakery_final_classification.csv"
FINAL_BAKERY_PATH = VERIFICATION_FOLDER / "bakery_final_list.csv"


REVIEW_01_PATH = MANUAL_REVIEW_FOLDER / "01_unclear_bakery_name_review.xlsx"
REVIEW_01B_PATH = MANUAL_REVIEW_FOLDER / "01b_unclear_bakery_word_address_review.xlsx"
REVIEW_02_PATH = MANUAL_REVIEW_FOLDER / "02_not_bakery_bakery_words_review.xlsx"
REVIEW_03_PATH = MANUAL_REVIEW_FOLDER / "03_bakery_reason_review.xlsx"
REVIEW_03B_PATH = MANUAL_REVIEW_FOLDER / "03b_bakery_physical_premise_review.xlsx"

# Repeated Lists

In [4]:
REVIEW_COLUMNS = [
    "BusinessNameClean",
    "BusinessName",
    "BusinessType",
    "Address",
    "PostCode",
    "LocalAuthorityName",
    "BakeryRank",
    "BakeryScore",
    "AIVerdict",
    "AIReason"
]

VALID_MANUAL_VERDICT = [
    "BAKERY",
    "NOT_BAKERY",
    "UNCLEAR",
    "INACTIVE"]

# Review-file saving function 

Since the review will occur in steps, and the AI verification is being done in stages due to API rate limits - it is worthwhile creating a function that will allow us to save and update our manual reviews at any state of review or in the case that more data comes in for us to review.

In [1]:
def update_review_file(review_df, review_path):

    review_df = (review_df
                 .sort_values("BakeryRank")
                 .reset_index(drop=True))


    # Create the file if it does not exist
    if not review_path.exists():
        review_df.to_excel(review_path, index=False)

        print(f"Created {review_path.name} with {len(review_df)} rows.")

        return

    # Otherwise compare with the existing manual review
    existing_review = pd.read_excel(review_path)

    new_rows = review_df[~review_df["BakeryRank"]
                         .isin(existing_review["BakeryRank"])
                         ].copy()

    # Do not touch the workbook if nothing has changed
    if len(new_rows) == 0:
        print(f"{review_path.name}: unchanged with ({len(existing_review)} rows preserved).")

        return

    # To prevent overwrites, a backup is created before changing existing workbooks
    backup_path = review_path.with_name(
        f"{review_path.stem}_backup_{pd.Timestamp.now():%Y%m%d_%H%M%S}{review_path.suffix}")

    copy2(review_path, backup_path)

    # Append only genuinely new businesses
    updated_review = pd.concat([existing_review, new_rows], ignore_index=True)

    updated_review = (updated_review
                      .sort_values("BakeryRank")
                      .reset_index(drop=True))

    assert not updated_review["BakeryRank"].duplicated().any()

    updated_review.to_excel(review_path, index=False)

    print(f"Backed up existing review and added {len(new_rows)} new rows.")

    print(f"There are {len(updated_review)} rows now in {review_path.name}.")

# Manual-review application

After manual review, is committed on the excel file this needs to be applied to the bakery list with any changes noted and businesses marked for having been checked and reviewed.

In [5]:
def apply_manual_review(results, review_path, review_source):

    review = pd.read_excel(review_path)

    assert not review["BakeryRank"].duplicated().any()

    review["ManualVerdict"] = (review["ManualVerdict"]
                               .fillna("")
                               .str.strip()
                               .str.upper()
                               .str.replace(" ", "_", regex=False))

    review["ReviewNote"] = (review["ReviewNote"].fillna(""))

    # Some reviews do not create changes and only need to be checked so this has been added
    if "Checked" in review.columns:
        checked = (review["Checked"]
                   .fillna("")
                   .astype(str)
                   .str.strip()
                   .str.upper()
                   .eq("X"))

        completed_review = review[checked].copy()
        pending = (~checked).sum()

        assert completed_review["ManualVerdict"].isin(VALID_MANUAL_VERDICT).all()

    # The initial check required a Manual verdict to be written
    else:
        completed_review = review[
            review["ManualVerdict"].isin(VALID_MANUAL_VERDICT)].copy()

        pending = (len(review) - len(completed_review))

    # The inactive flags should be left with a similar note
    inactive_rows = (completed_review["ManualVerdict"] == "INACTIVE")

    completed_review.loc[inactive_rows & completed_review["ReviewNote"].eq(""), 
                         "ReviewNote"] = "Excluded because business was inactive, closed or dissolved."

    completed_review.loc[inactive_rows, "ManualVerdict"] = "NOT_BAKERY"


    updates = completed_review.set_index("BakeryRank")
    rows = results["BakeryRank"].isin(updates.index)

    assert rows.sum() == len(completed_review)

    # Manual review queues should not overlap
    assert not (rows & results["Reviewed"]
                ).any(), f"{review_source} overlaps an earlier manual review."


    results.loc[rows, "FinalVerdict"] = (
        results.loc[rows, "BakeryRank"]
        .map(updates["ManualVerdict"]))

    results.loc[rows, "ReviewNote"] = (
        results.loc[rows, "BakeryRank"]
        .map(updates["ReviewNote"]))

    results.loc[rows, "Reviewed"] = True

    results.loc[rows, "ReviewSource"] = review_source

    return len(completed_review), pending

# Loading and inspecting AI verification

In [6]:
ai_results = pd.read_csv(AI_RESULTS_PATH)

ai_results = (ai_results
              .sort_values("BakeryRank")
              .reset_index(drop=True))

print(f"AI verification results loaded: {len(ai_results)}")

AI verification results loaded: 28847


In [7]:
print(f"AI verdicts:\n{ai_results['AIVerdict'].value_counts()}")

print(f"\nFHRS business types:\n{ai_results['BusinessType'].value_counts()}")

AI verdicts:
AIVerdict
NOT_BAKERY    19875
UNCLEAR        6029
BAKERY         2943
Name: count, dtype: int64

FHRS business types:
BusinessType
Restaurant/Cafe/Canteen                  7578
Retailers - other                        5860
Other catering premises                  4766
Caring Premises                          2445
Takeaway/sandwich shop                   1880
School/college/university                1824
Pub/bar/nightclub                        1298
Mobile caterer                           1101
Manufacturers/packers                     702
Hotel/bed & breakfast/guest house         565
Distributors/Transporters                 369
Retailers - supermarkets/hypermarkets     271
Importers/Exporters                       174
Farmers/growers                            14
Name: count, dtype: int64


# Excluding INACTIVE from manual review

In the prompt we created for the AI verfication, we requested any businesses that seemed to be inactive, closed or dissolved to be highlighted with INACTIVE: as the AIReason input. These should be removed at this stage from the manual review and can later be removed from the final verified bakery set.

In [9]:
inactive_results = ai_results[ai_results["AIReason"]
                              .fillna("")
                              .str.startswith("INACTIVE:")].copy()

print(f"Businesses identified as inactive: {len(inactive_results)}")

print(f"\nAI verdicts for inactive businesses:\n{inactive_results['AIVerdict'].value_counts()}")

Businesses identified as inactive: 166

AI verdicts for inactive businesses:
AIVerdict
NOT_BAKERY    151
UNCLEAR         8
BAKERY          7
Name: count, dtype: int64


In [13]:
(inactive_results[[
    "BusinessName",
    "BusinessType",
    "Address",
    "BakeryRank",
    "AIVerdict",
    "AIReason"]]).head(20)

,BusinessName,BusinessType,Address,BakeryRank,AIVerdict,AIReason
611,Percy Ingle Bakeries,Restaurant/Cafe/Canteen,"231 WELL STREET, Hackney, London",612,NOT_BAKERY,INACTIVE: The Percy Ingle bakery chain permanently closed all its locations by March 2021.
619,Roll N Bake Ltd,Other catering premises,NaN,620,UNCLEAR,"INACTIVE: Companies House shows ""Roll N Bake Ltd"" is in the process of voluntary strike-off."
1304,Dum Treat Ltd,Other catering premises,NaN,1305,NOT_BAKERY,INACTIVE: Companies House shows Dum Treat Ltd was dissolved in 2019 and its business activity was manufacturing dietetic food.
1361,Creme delights ltd,Restaurant/Cafe/Canteen,"154, Old Kent Road, London",1362,NOT_BAKERY,INACTIVE: Companies House shows the company is dissolved. Uber Eats menu shows it was a dessert shop selling milkshakes and cake slices.
1459,Swely's Sweethouse Ltd,Other catering premises,NaN,1460,NOT_BAKERY,INACTIVE: Swely's Sweethouse Ltd is dissolved and its business activity is not specified as bakery.
2010,Happi Cheesecakes,Other catering premises,NaN,2011,BAKERY,INACTIVE: Website states it is undergoing rebrand but previously made cheesecakes.
2446,cake cult london,Restaurant/Cafe/Canteen,"HACKNEY BRIDGE ECHO BUILDINGS EAST BAY LANE, Hackney, London",2447,BAKERY,"INACTIVE: Cake Cult London was a vegan bakery offering plant-based cakes, pastries, brownies, and cookies."
2635,Karen's Cake House,Other catering premises,NaN,2636,BAKERY,INACTIVE: Karen's Cake House specialized in baking and decorating bespoke cakes for special occasions.
2841,Salmon News Ltd,Retailers - other,"124 Salmon Lane, London",2842,NOT_BAKERY,"INACTIVE: Companies House lists Salmon News Ltd as dissolved, and it was a newsagent."
3172,Whirlylicious LTD,Retailers - other,NaN,3173,NOT_BAKERY,"INACTIVE: Whirlylicious LTD creates candy floss cakes, which are not baked goods."


In [14]:
inactive_ranks = set(inactive_results["BakeryRank"])

review_data = ai_results[~ai_results["BakeryRank"].isin(inactive_ranks)].copy()

print(f"Businesses remaining for review: {len(review_data)}")

Businesses remaining for review: 28681


# Reviewing UNCLEAR Bakery classification

In [15]:
unclear_results = review_data[review_data["AIVerdict"] == "UNCLEAR"].copy()

print(f"Active UNCLEAR businesses: {len(unclear_results)}")

Active UNCLEAR businesses: 6021


In [16]:
unclear_results[[
        "BusinessName",
        "BusinessType",
        "Address",
        "BakeryRank",
        "BakeryScore",
        "AIReason"
    ]].sort_values("BakeryRank").head(30)

,BusinessName,BusinessType,Address,BakeryRank,BakeryScore,AIReason
0,Bakery And Cake,Other catering premises,NaN,1,0.999579,The name is too generic and the partial postcode matches multiple businesses.
2,Bakers Cakes,Restaurant/Cafe/Canteen,NaN,3,0.999565,"Multiple businesses match the name and partial postcode, with no clear London match."
12,A?fe Bakery,Other catering premises,NaN,13,0.997925,"The name is too generic, and no specific business with ""A?fe Bakery"" in EN1 could be identified."
13,Olgas Sweets Bakery,Retailers - other,NaN,14,0.997696,"No specific business named ""Olgas Sweets Bakery"" in TW3 could be identified."
14,Sugar's Bakery,Restaurant/Cafe/Canteen,NaN,15,0.996937,"Multiple businesses with similar names exist, and no definitive ""Sugar's Bakery"" in TW1 was found."
16,Sourdough Delights Bakery,Retailers - other,NaN,17,0.996723,"Multiple businesses with ""Sourdough Delights"" exist, and no specific one in DA15 could be confirmed."
18,Sweetwater Bakery,Other catering premises,NaN,19,0.996324,"The name is too generic, and no specific business with ""Sweetwater Bakery"" in W10 could be identified."
22,Nou Bakery,Other catering premises,NaN,23,0.995919,"The name ""Nou Bakery"" is generic, and search results point to multiple businesses with conflicting locations."
31,1K1V Ltd,Retailers - other,"Ealing Christian Centre, 268 Northfield Avenue, Ealing",32,0.995045,No specific business activity found for this company at the given address.
32,Cake baking,Other catering premises,NaN,33,0.994930,"""Cake baking"" is a generic term, and no specific business with this name was found in SE22."


## Review 01 - UNCLEAR businesses with "bakery" in the name

Looking at the highest-ranked UNCLEAR results showed mostly businesses containing "bakery" in their name, this seemed suspicious as a lot of them were nontheless deemed UNCLEAR by the AI. The first review priortises these businesses since the AI was unable to confirm their identity or activity.

In [17]:
review_01 = unclear_results[unclear_results["BusinessName"]
                            .fillna("")
                            .str.lower()
                            .str.contains("bakery", regex=False)].copy()

print(f"Review 01 businesses: {len(review_01)}")

Review 01 businesses: 125


In [27]:
review_01 = (review_01[REVIEW_COLUMNS]
             .sort_values("BakeryRank")
             .copy())

review_01["ManualVerdict"] = ""
review_01["ReviewNote"] = ""

update_review_file(review_01, REVIEW_01_PATH)

01_unclear_bakery_name_review.xlsx: unchanged with (125 rows preserved).


## Further exploring businesses without "bakery" in UNCLEAR

The first review showed that most of the time businesses without an FHRS address ended up not being a bakery or being harder to review. The next review should scope outside of businesses explicitly named bakeries while prioritsing those with a recorded address.

In [18]:
bakery_words = ["bakery", "baker", "bake", "bakes", "cake", "cakes", "bread", "pastry", "patisserie", "cookie",
                "cookies", "cupcake", "sourdough", "doughnut", "donut", "bagel", "baklava", "pie", "biscuit", "brownie",
                "muffin", "croissant", "focaccia", "brioche", "pretzel", "gingerbread", "macaron"]

bakery_word_pattern = "|".join(bakery_words)

In [19]:
review_01_ranks = set(review_01["BakeryRank"])

remaining_unclear = unclear_results[~unclear_results["BakeryRank"].isin(review_01_ranks)].copy()

bakery_word_unclear = remaining_unclear[remaining_unclear["BusinessName"]
                                        .fillna("")
                                        .str.lower()
                                        .str.contains(bakery_word_pattern)].copy()

print(f"Other bakery-word related UNCLEAR businesses: {len(bakery_word_unclear)}"
)

Other bakery-word related UNCLEAR businesses: 706


These can be tackled in order of priority, first those with an address.

In [20]:
has_address = (bakery_word_unclear["Address"]
               .fillna("")
               .str.strip()
               .ne(""))

print(f"With an FHRS address: {has_address.sum()}")

print(f"Without an FHRS address: {(~has_address).sum()}")

With an FHRS address: 70
Without an FHRS address: 636


## Review 01b - Bakery-word related UNCLEAR businesses with an FHRS address

In total there are 706 UNCLEAR business with bakery-related names, but only 70 businesses had a recorded address. Those without an address become unreliable when it comes to identification, so the first next target will be those with a recorded address.

In [22]:
review_01b = bakery_word_unclear[has_address].copy()

print(f"Review 01b businesses: {len(review_01b)}")

review_01b = (review_01b[REVIEW_COLUMNS + ["LocationMatch"]]
              .sort_values("BakeryRank")
              .copy())

review_01b["ManualVerdict"] = ""
review_01b["ReviewNote"] = ""

update_review_file(review_01b, REVIEW_01B_PATH)

Review 01b businesses: 70
Created 01b_unclear_bakery_word_address_review.xlsx with 70 rows.


# Reviewing NOT_BAKERY classifications

In [23]:
not_bakery_results = review_data[review_data["AIVerdict"] == "NOT_BAKERY"].copy()

print(f"Active NOT_BAKERY businesses: {len(not_bakery_results)}")

Active NOT_BAKERY businesses: 19724


In [24]:
not_bakery_results[[
    "BusinessName",
    "BusinessType",
    "Address",
    "BakeryRank",
    "BakeryScore",
    "AIReason"
]].sort_values("BakeryRank").head(30)

,BusinessName,BusinessType,Address,BakeryRank,BakeryScore,AIReason
5,B.Bakery,Mobile caterer,"Unit 16 Skylines Village, Limeharbour, London",6,0.999023,"B.Bakery is associated with Mercy to Humanity, a charity providing humanitarian meals."
8,B.J Bakery,Retailers - other,"15 Goresbrook Road, Dagenham",9,0.998572,"BJ Bakery in Dagenham is a retailer of other foods, including sandwiches and rolls."
57,La Mira Sweet Treat Bakery,Other catering premises,NaN,58,0.989967,"La Mira Sweet Treat specializes in chocolate-coated Oreos and other personalized desserts, not traditional baked goods."
65,Muzda Bakery,Retailers - other,"129 Green Street, Forest Gate, London",66,0.988255,"While it sells baked pastries and biscuits, it also offers a wide selection of Indian snacks and frozen products."
117,Malaki's Cakes Bread & Pastries,Takeaway/sandwich shop,"38 Westow Hill, Upper Norwood, London",118,0.980307,Malaki's Cakes Bread & Pastries is primarily a Caribbean takeaway that also sells bread and cakes.
126,Rose Bakery,Restaurant/Cafe/Canteen,"18 - 21 HAYMARKET, LONDON, United Kingdom",127,0.978985,"Rose Bakery is a cafe within Dover Street Market, serving light meals, coffee, and some cakes."
127,Ishter bakery,Restaurant/Cafe/Canteen,"141 Greenford Road, Harrow",128,0.978948,"Ishter Bakery at this address appears to be Ishtar Grill, a Middle Eastern restaurant."
129,Dalpash Bakery,Restaurant/Cafe/Canteen,"228 Horn Lane, Acton",130,0.978783,Dalpash Bakery is a Lebanese bakery and BBQ house primarily offering Lebanese dishes and pizza.
181,Tasty Bakery,Restaurant/Cafe/Canteen,"Unit 10, Bellingham Trading Estate, Franthorne Way, London",182,0.969305,"Companies House shows ""Tasty Bakery Limited"" is dissolved, and other results are for ""Flakey Crust Bakery"" at the same address, which sells Jamaican patties and bread."
183,Cacao Bakery,Manufacturers/packers,NaN,184,0.968922,"Cacao Bakery is registered in Ealing, but ""Kova Patisserie Ealing"" is a Japanese-French patisserie at a different address."


## Review 02 - Bakery-word businesses classified NOT_BAKERY

Looking through the highest-ranked NOT_BAKERY results showed quite a few businesses with clearly bakery-related names. The first review that seems obvious for the NOT_BAKERY classification is using the bakery-related word list created earlier and potentially using those terms to find false negatives in a manual review.

In [25]:
review_02 = not_bakery_results[not_bakery_results["BusinessName"]
                               .fillna("")
                               .str.lower()
                               .str.contains(bakery_word_pattern)].copy()

print(f"Review 02 businesses: {len(review_02)}")

Review 02 businesses: 222


In [26]:
review_02 = (review_02[REVIEW_COLUMNS]
             .sort_values("BakeryRank")
             .copy())

review_02["ManualVerdict"] = ""
review_02["ReviewNote"] = ""

update_review_file(review_02, REVIEW_02_PATH)

02_not_bakery_bakery_words_review.xlsx: unchanged with (222 rows preserved).


# Reviewing BAKERY classifications

In [28]:
bakery_results = review_data[review_data["AIVerdict"] == "BAKERY"].copy()

print(f"Active BAKERY businesses: {len(bakery_results)}")

Active BAKERY businesses: 2936


In [29]:
bakery_results["BusinessType"].value_counts()

BusinessType
Restaurant/Cafe/Canteen                  899
Other catering premises                  831
Retailers - other                        648
Manufacturers/packers                    234
Takeaway/sandwich shop                   232
Mobile caterer                            71
Caring Premises                            6
Distributors/Transporters                  5
Retailers - supermarkets/hypermarkets      3
Hotel/bed & breakfast/guest house          3
School/college/university                  2
Pub/bar/nightclub                          2
Name: count, dtype: int64

## Exploring unusual BAKERY classifications

Majority of the businesses classified as a BAKERY shared a handfull of Business types that could reaonably be related to bakery activity. A small number of them belonged to less typical categories and this is worth reviewing as our first review on the BAKERY classified set.

In [30]:
unusual_bakery_types = [
    "Caring Premises",
    "School/college/university",
    "Retailers - supermarkets/hypermarkets",
    "Distributors/Transporters",
    "Hotel/bed & breakfast/guest house",
    "Importers/Exporters",
    "Farmers/growers",
    "Pub/bar/nightclub"
]

unusual_bakeries = bakery_results[bakery_results["BusinessType"].isin(unusual_bakery_types)].copy()

print(f"BAKERY businesses with unusual FHRS types: {len(unusual_bakeries)}")

BAKERY businesses with unusual FHRS types: 21


In [31]:
unusual_bakeries[[
    "BusinessName",
    "BusinessType",
    "BakeryRank",
    "BakeryScore",
    "AIReason"
]]

,BusinessName,BusinessType,BakeryRank,BakeryScore,AIReason
66,Memleket Limited Bakery,Distributors/Transporters,67,0.988218,"Companies House and business directories list its activities as manufacturing bread, pastries, and cakes."
727,Bread Tree,Distributors/Transporters,728,0.816227,"Identified as a bakery in Croydon, specializing in bread."
1037,Hart & Lova Bakery,Caring Premises,1038,0.732731,"Official website and business listings describe it as a bakery and patisserie known for pastries, breads, and cakes."
1630,Julie Scrumptious Cakes,Caring Premises,1631,0.560498,Official website and listings confirm it makes and sells custom cakes and other baked goods.
2988,HOLY SUGAR,School/college/university,2989,0.231708,"Holy Sugar is a pastry chef-led business specializing in American-style sweet pies, brownies, and cheesecakes."
3731,The Tasty Treats,Caring Premises,3732,0.151259,The business is a micro bakery specializing in custom celebration cakes and other baked treats.
3911,Cookie Jar London Hammersmith,Retailers - supermarkets/hypermarkets,3912,0.142285,Official website and delivery platforms confirm it sells freshly baked cookies and cookie cakes.
4739,Natoora ltd,Distributors/Transporters,4740,0.106959,"Natoora Ltd. has a bakery called Alma, which is part of their food system business."
5638,Dulce Limited,Retailers - supermarkets/hypermarkets,5639,0.083113,"Dulce Limited operates as a food and cake shop, offering Eastern European food, drink, and cakes."
8843,Datekin And Tuffnut,Distributors/Transporters,8844,0.046492,"Datekin specializes in gourmet date snacks, including date cakes and date balls, and refers to an ""artisanal bakery""."


In [32]:
bakery_results[[
    "BusinessName",
    "BusinessType",
    "BakeryRank",
    "BakeryScore",
    "AIReason"
]].sort_values("BakeryScore").head(30)

,BusinessName,BusinessType,BakeryRank,BakeryScore,AIReason
28799,H. R. Higgins,Retailers - other,28800,0.015020,It is a coffee and tea specialist with a downstairs cafe offering freshly baked cakes and pastries.
28766,Evelina's Lab,Other catering premises,28767,0.015038,"Evelina's Patisserie specializes in macarons, cakes, and French pastries."
28754,The Munchin,Restaurant/Cafe/Canteen,28755,0.015043,"The menu offers homemade baked goods like banana bread, cookies, and brownies."
28748,Tai Pan,Retailers - other,28749,0.015050,Multiple sources confirm it is a Chinese bakery selling various baked goods.
28694,Buongiorno,Restaurant/Cafe/Canteen,28695,0.015075,"The business is a patisserie, coffee, and gelato shop selling cakes, pastries, and focaccia."
28628,Hotteok,Mobile caterer,28629,0.015102,"Hotteok specializes in Korean pancakes, which are a type of baked product."
28555,Kosher Central London,Retailers - other,28556,0.015141,"It is a kosher grocery and provisions shop offering fresh baked challah, pastry, bagels, and sourdough."
28534,Kouttone,Other catering premises,28535,0.015153,Social media shows it makes sourdough raisin brioche and other French viennoiseries.
28519,Doctor's Orders,Restaurant/Cafe/Canteen,28520,0.015157,This cafe within a Dr. Martens store serves baked goods from other London bakeries.
28474,Willow's Delish,Restaurant/Cafe/Canteen,28475,0.015180,"Willow's Delish is a cafe run by a school, with students developing baking skills and selling cakes."


## Review 03 - BAKERIES with weak, conflicting or unusual evidence

Taking a look at some of the low scoring BAKERY classifications as well as looking at the types of wording used in the AI reasons where the business looks suspicious, could allow us to parse the reasoning for keywords that show reasonable doubt. This combined with the unusual FHRS business types might be a good first manual review of the bakery classifications.

In [33]:
review_reason_words = [
    "name suggests",
    "name indicates",
    "business name",
    "likely",
    "imply",
    "no conflicting evidence",
    "food and beverage business",

    "primarily a restaurant",
    "primarily a cafe",
    "primarily a café",
    "primarily a takeaway",
    "primarily a dessert",
    "dessert shop",
    "dessert spot",
    "catering service",

    "bakery section",
    "incidental",
    "limited bakery",
    "not a bakery",
    "not traditional bakery",
    "not baked goods",
    "not bakery products",

    "baking club",
    "baking workshop",
    "after-school",
    "cake decorating supplies",
    "supplies and equipment"]

review_reason_pattern = "|".join(review_reason_words)

weak_reason_bakeries = bakery_results[bakery_results["AIReason"]
                                      .fillna("")
                                      .str.lower()
                                      .str.contains(review_reason_pattern)].copy()

print(f"BAKERIES with weak or conflicting reasons: {len(weak_reason_bakeries)}")

BAKERIES with weak or conflicting reasons: 92


In [34]:
review_03 = pd.concat([weak_reason_bakeries, unusual_bakeries])

review_03 = (review_03
             .drop_duplicates(subset="BakeryRank")
             .sort_values("BakeryRank")
             .reset_index(drop=True))

print(f"Review 03 businesses: {len(review_03)}")

Review 03 businesses: 111


In [35]:
review_03 = (review_03[REVIEW_COLUMNS]
             .sort_values("BakeryRank")
             .copy())

review_03["ManualVerdict"] = "BAKERY"
review_03["Checked"] = ""
review_03["ReviewNote"] = ""

update_review_file(review_03, REVIEW_03_PATH)

03_bakery_reason_review.xlsx: unchanged with (111 rows preserved).


## Further exploring physical bakery provision

Many of the BAKERY classifications seemed to have been identifed by the AI as "home-based", "residential" or "operating from a private address". Though these can be legitimate bakeries and businesses, this shouldn't be reason enough from the AI to assume that these are running businesses that are representing bakery provision adequaltely and are worht a separate check.

In [36]:
physical_premises_words = [
    "private address",
    "residential",
    "home-based",
    "home based",
    "home baker",
    "from home",
    "domestic premises",
    "home business"
]

physical_premises_pattern = "|".join(physical_premises_words)

review_03_ranks = set(review_03["BakeryRank"])

review_03b = bakery_results[bakery_results["AIReason"]
                            .fillna("")
                            .str.lower()
                            .str.contains(physical_premises_pattern)
                            & (~bakery_results["BakeryRank"].isin(review_03_ranks))].copy()

print(f"Review 03b businesses: {len(review_03b)}")

Review 03b businesses: 100


In [39]:
review_03b = (review_03b[REVIEW_COLUMNS]
              .sort_values("BakeryRank")
              .copy())

review_03b["ManualVerdict"] = "BAKERY"
review_03b["Checked"] = ""
review_03b["ReviewNote"] = ""

update_review_file(review_03b, REVIEW_03B_PATH)

03b_bakery_physical_premise_review.xlsx: unchanged with (100 rows preserved).
